In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

In [ ]:
# open the timeseries file
ds = xr.open_dataset('filename')
display(ds)

# check the first time to see if an offset is needed
ds.time.values[0]

# apply the time offset if needed
adjusted_time = pd.to_datetime(ds['time'].values) + pd.DateOffset(years=, months=, days=)
ds = ds.assign_coords(time=('sg_data_point', adjusted_time))

# check time again to make sure the adjustment took and makes sense
ds.time.values[0]

In [ ]:
# check the depth to see if an offset is needed
depth = ds.depth.values
time = ds.time.values
plt.plot(time, -depth)

# is depth above 0? is a linear or variable offset needed? if linear, add a value to all depth values

In [ ]:
# if variable, apply this offset
depth = ds.depth.values
dive_ids = ds.sg_data_point_dive_number.values

# stores the corrected depth
depth_corrected = depth.copy()

# unique dives in order
unique_dives = sorted(set(dive_ids))

# loop to get and apply offset
for d in unique_dives:
    # indices in sg_data_point belonging to this dive
    idx = (dive_ids == d)

    # first depth measurement in this dive
    offset = depth[idx][0]

    # apply correction
    depth_corrected[idx] = depth[idx] - offset

ds["depth_corrected"] = (("sg_data_point",), depth_corrected)

depth_sg = ds['depth_corrected']
time_sg = ds['time']

# check to see if offset worked
plt.plot(time_sg, -depth_sg)

In [ ]:
# SG Data Point Files

# make sure you are where you want to be
print(os.getcwd())

# rename variables for clarity
ds['PAR_470nm'] = ds['eng_wlbb2fl_sig470nm']
ds['particle_concentration_700nm'] = ds['eng_wlbb2fl_sig700nm']
ds['chlorophyll_695nm'] = ds['eng_wlbb2fl_sig695nm']
ds['dissolved_oxygen'] = ds['aanderaa4330_dissolved_oxygen']
ds['instrument_dissolved_oxygen'] = ds['aanderaa4330_instrument_dissolved_oxygen']

# add metadata
ds['PAR_470nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig470nm'
ds['particle_concentration_700nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig700nm'
ds['chlorophyll_695nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig695nm'
ds['dissolved_oxygen'].attrs['pre_cleaning_name'] = 'aanderaa4330_dissolved_oxygen'
ds['instrument_dissolved_oxygen'].attrs['pre_cleaning_name'] = 'aanderaa4330_instrument_dissolved_oxygen'

ds = ds.assign_coords(time=('sg_data_point', adjusted_time))

#Select the relevant variables (depth_corrected if correction applied, just depth if not)
new_ds = ds[['time', 'depth_corrected', 'latitude', 'longitude','temperature', 'salinity', 'dissolved_oxygen', 'instrument_dissolved_oxygen', 'PAR_470nm', 'particle_concentration_700nm', 'chlorophyll_695nm']]

#Convert to DataFrame and save as csv and nc 
new_ds.to_dataframe().reset_index().to_csv('newfilename.csv', index=False)
new_ds.to_netcdf('newfilename.nc')
display(new_ds)

In [ ]:
# Depth Average Currents Files

# did you apply a time offset before? if so, apply the same one here
adjusted_start = pd.to_datetime(ds['start_time'].values) + pd.DateOffset(years=, months=, days=)
adjusted_end = pd.to_datetime(ds['end_time'].values) + pd.DateOffset(years=, months=, days=)
ds = ds.assign_coords(start_time=('dive', adjusted_start), end_time=('dive', adjusted_end))

# rename variables for clarity
ds['U_DAC'] = ds['depth_avg_curr_east']
ds['V_DAC'] = ds['depth_avg_curr_north']

# add metadata
ds['U_DAC'].attrs['pre_cleaning_name'] = 'depth_avg_curr_east'
ds['V_DAC'].attrs['pre_cleaning_name'] = 'depth_avg_curr_north'

#Select the relevant variables
new_ds = ds[['U_DAC', 'V_DAC', 'start_time', 'end_time', 'start_latitude', 'end_latitude', 'start_longitude', 'end_longitude']]
display(new_ds)

#Convert to DataFrame and save as csv and nc 
new_ds.to_dataframe().reset_index().to_csv('DAC_newfilename.csv', index=False)
new_ds.to_netcdf('DAC_newfilename.nc')
